# Python lab: Martingale Pricing

ใช้ Python 3 แล้วเลือก **Run All** ไม่ต้องติดตั้งแพ็กเกจเพิ่ม ส่วนแรกเก็บข้อความและสมการจากบทเรียนฉบับเดียวกับเว็บไซต์ ส่วนท้ายเป็นการทดลองที่รันได้ครบในไฟล์นี้ ข้อมูลทั้งหมดเป็นตัวอย่างสมมติ การสุ่มระบุ seed ของ Python จึงไม่จำเป็นต้องได้ตัวเลขรายเส้นทางตรงกับเว็บไซต์

[เปิดบทเรียน](https://nutdnuy.github.io/quantitative-finance-notes/martingale-pricing.html)

<h1 id="martingale-pricing-title">Martingale Pricing — Black–Scholes อีกมุมหนึ่ง</h1>

ทำไมราคา Option จึงคำนวณจากค่าเฉลี่ยได้ และต้องเฉลี่ยภายใต้ความน่าจะเป็นแบบไหน?

ในบท [Black–Scholes Model](https://nutdnuy.github.io/quantitative-finance-notes/black-scholes-model.html) เราใช้ Delta hedge ตัดความเสี่ยงจากการขยับของหุ้น แล้วใช้ no-arbitrage สร้างสมการราคา บทนี้จะเดินเข้าหาผลลัพธ์เดียวกันผ่าน **ความน่าจะเป็น** โดยอธิบายว่าทำไมการคิดลด payoff คาดหมายจึงให้ราคาที่สอดคล้องกับพอร์ตเลียนแบบ

หัวใจคือการเลือกทั้ง **หน่วยที่ใช้วัดมูลค่า** และ **มาตรวัดความน่าจะเป็น** ให้สอดคล้องกัน เมื่อใช้บัญชีเงินสดเป็นหน่วยวัดและใช้มาตรวัด Q ที่เหมาะสม มูลค่าพอร์ตเลียนแบบจะเป็น martingale เราจึงถอยจาก payoff วันหมดอายุกลับมาหาราคาวันนี้ได้

อ่านโดยมีพื้นฐาน [Itô’s lemma](https://nutdnuy.github.io/quantitative-finance-notes/applied-stochastic-calculus.html#ito-lemma) และ [พอร์ต self-financing](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#self-financing) หากยังไม่คุ้นกับสัญลักษณ์ความน่าจะเป็น ให้เริ่มจากตัวอย่างสองสถานะและห้องทดลอง แล้วค่อยกลับมาอ่าน Girsanov ทุกตัวเลขในบทเป็นตัวอย่างสมมติ ไม่ใช่ราคาตลาดจริง

<a id="pricing-question"></a>

## ค่าเฉลี่ยเดียวกัน แต่ใช้คนละน้ำหนัก

ทบทวนตัวอย่างหุ้น 100 ที่อีกหนึ่งวันเป็น 101 หรือ 99, Call มี strike 100 และดอกเบี้ย 0% ถ้าเชื่อว่าราคาขึ้นด้วยความน่าจะเป็นจริง p=0.6 จะได้ payoff คาดหมาย 0.6 แต่พอร์ตหุ้นครึ่งหน่วยกับเงินกู้ 49.5 เลียนแบบ payoff 1/0 ได้ด้วยต้นทุน **0.5**

$$
V_0=0.5(100)-49.5=0.5,\qquad
V_T=0.5S_T-49.5\in\{1,0\}.
$$

น้ำหนักสำหรับตั้งราคาจึงเป็น q=0.5 ซึ่งทำให้ราคาหุ้นคาดหมายเท่ากับ 100 และ payoff คาดหมายเท่ากับ 0.5 การเปลี่ยน p ไม่ได้เปลี่ยนต้นทุนพอร์ตที่ให้ผลลัพธ์เดียวกันทุกสถานะ ดูขั้นตอนใน [Binomial Model](https://nutdnuy.github.io/quantitative-finance-notes/binomial-model.html)

ในเวลาต่อเนื่อง เราเรียกมาตรวัดที่อธิบายความน่าจะเป็นจริงว่า **P** และมาตรวัดสำหรับตั้งราคาที่ใช้บัญชีเงินสดเป็นหน่วยวัดว่า **Q** ส่วน q ตัวเล็กยังหมายถึงน้ำหนักขึ้นในต้นไม้ Binomial ตามเดิม

| สิ่งที่ถาม | มาตรวัดที่ใช้ | สิ่งที่ได้ |
|---|---|---|
| ภายใต้แบบจำลองจริง หุ้นหรือ P&L มีโอกาสเป็นอย่างไร? | P | การคาดการณ์ การประเมินความเสี่ยง |
| payoff ที่เลียนแบบได้ควรมีราคาเท่าไร? | Q ที่สอดคล้องกับ numeraire | มูลค่าที่สอดคล้องกับ no-arbitrage |

Q ไม่ได้แปลว่าผู้ลงทุนทุกคนไม่กลัวความเสี่ยง และไม่ได้บอกให้แก้สมมติฐานผลตอบแทนจริงของหุ้นเป็น r

<a id="market-and-information"></a>

## ตลาดเล็ก ๆ และข้อมูลที่รู้ ณ เวลา t

ให้ \(\mathcal F_t\) เป็นข้อมูลที่ทราบถึงเวลา t เช่น เส้นทางราคาที่เกิดขึ้นแล้ว ลำดับของชุดข้อมูลนี้เรียกว่า [filtration](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#filtration) กลยุทธ์ซื้อขายต้องใช้ข้อมูลที่มีอยู่ขณะตัดสินใจ ไม่ใช้ราคาที่จะเกิดในอนาคต

เริ่มด้วยหุ้นไม่มีปันผลและบัญชีเงินสด:

$$
dB_t=rB_t\,dt,\quad B_0=1,\quad B_t=e^{rt},
$$

$$
dS_t=\mu S_t\,dt+\sigma S_t\,dW_t^{P}.
$$

S คือราคาหุ้นต่อหน่วย, B คือมูลค่าบัญชีเงินสดเริ่มต้นหนึ่งหน่วย, r คือดอกเบี้ยทบต้นต่อเนื่องต่อปี, μ คืออัตราผลตอบแทนคาดหมายของหุ้นต่อปี และ σ คือ volatility ต่อรากปี เราวัดเวลาเป็นปี โดย \(\tau=T-t\) เป็นเวลาที่เหลือจน Option หมดอายุ

สมมติให้ r, μ, σ คงที่, σ>0, ซื้อขายได้ต่อเนื่องและเป็นเศษหน่วยได้, short และกู้ยืมได้, ไม่มีค่าธรรมเนียมหรือข้อจำกัดสภาพคล่อง และตลาดไม่มี arbitrage ใช้ filtration ที่สร้างจาก Brownian motion แหล่งเดียว สำหรับส่วนขยายท้ายบทจะเปลี่ยนสมมติฐานบางข้ออย่างชัดเจน

สัญญา European จ่าย \(H=G(S_T)\) ที่เวลา T เช่น Call จ่าย \((S_T-K)^+\) โดย \(x^+=\max(x,0)\) ค่า H เป็น **payoff** ยังไม่ได้หัก premium หรือดอกเบี้ยของเงินที่ใช้ซื้อ Option

<a id="self-financing-pricing"></a>

## Self-financing: เปลี่ยนจำนวนหุ้นได้ แต่เงินต้องมาจากในพอร์ต

ให้พอร์ตถือหุ้น \(\Delta_t\) หน่วย และบัญชีเงินสด \(\beta_t\) หน่วย:

$$
V_t=\Delta_t S_t+\beta_t B_t.
$$

เงื่อนไข self-financing คือ

$$
dV_t=\Delta_t\,dS_t+\beta_t\,dB_t.
$$

กำไรขาดทุนมาจากสินทรัพย์ที่ถืออยู่ เมื่อเพิ่มจำนวนหุ้น ต้องจ่ายด้วยเงินสดในพอร์ตหรือกู้เพิ่มในบัญชีเดียวกัน ตัวอย่างหุ้นราคา 100 เปลี่ยนจาก 0.50 เป็น 0.60 หน่วย ต้องนำเงินสด 10 ไปซื้อหุ้น มูลค่ารวม ณ ราคานั้นจึงไม่เพิ่มขึ้นเพราะการสับเปลี่ยน holdings

สมการนี้เป็น **เงื่อนไขของกลยุทธ์** ไม่ใช่การหาอนุพันธ์ของ \(\Delta_tS_t\) แล้วละทิ้งพจน์ \(d\Delta_t\) โดยไม่มีเหตุผล ทดลองการปรับหุ้นกับเงินสดได้ใน [ห้องทดลอง Delta hedge](https://nutdnuy.github.io/quantitative-finance-notes/black-scholes-model.html#discrete-hedging)

Arbitrage ในกรอบนี้คือกลยุทธ์ที่เริ่มด้วย V₀=0, ปลายทางไม่ขาดทุนด้วยความน่าจะเป็นหนึ่ง และมีโอกาสบวกที่จะได้กำไร เราจำกัดกลยุทธ์ให้เป็น **admissible** เช่น มูลค่าหลังคิดลดมีขอบเขตล่างตามเกณฑ์ที่กำหนด เพื่อไม่เปิดทางให้กลยุทธ์ทบเงินเดิมพันจนเป็นหนี้ได้ไม่จำกัด

<a id="discounted-martingale"></a>

## ทำไมต้องหารราคาด้วยบัญชีเงินสด

ราคาหุ้น 100 วันนี้กับ 100 ปีหน้าไม่ใช่มูลค่า ณ เวลาเดียวกัน เราจึงเปลี่ยนหน่วยวัดเป็นจำนวนหน่วยบัญชีเงินสด:

$$
\widetilde S_t=\frac{S_t}{B_t}=e^{-rt}S_t.
$$

ใช้ product rule โดย B ไม่มีส่วน Brownian จะได้

$$
d\widetilde S_t=(\mu-r)\widetilde S_t\,dt+
\sigma\widetilde S_t\,dW_t^{P}.
$$

การ discount ลบส่วนเติบโต r ออก แต่ยังเหลือ drift μ−r จึงไม่ได้ทำให้หุ้นเป็น martingale ภายใต้ P โดยอัตโนมัติ

[Martingale](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#martingale) คือกระบวนการที่มีค่าคาดหมายสัมบูรณ์จำกัด และค่าคาดหมายในอนาคตเมื่อใช้ข้อมูลปัจจุบันเท่ากับค่าปัจจุบัน:

$$
\mathbb E^{Q}[M_u\mid\mathcal F_t]=M_t,\qquad u\ge t.
$$

เส้นทางของ M ยังขึ้นลงได้มาก เงื่อนไขนี้พูดถึง **ค่าเฉลี่ยแบบมีเงื่อนไข** ภายใต้มาตรวัดที่ระบุ ไม่ได้หมายความว่าราคาแต่ละเส้นทางคงที่

ถ้า S₀=100, μ=12%, r=5% และเวลา 1 ปี:

$$
\mathbb E^{P}[e^{-rT}S_T]=100e^{0.07}\approx107.2508.
$$

เราจะหามาตรวัด Q ที่ทำให้ค่าเฉลี่ยหลังคิดลดเท่ากับ 100 ซึ่งเข้ากับการวัดด้วยบัญชีเงินสด

<a id="girsanov"></a>

## Girsanov: เปลี่ยนน้ำหนักของเส้นทาง

กำหนด [market price of risk](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#market-price-of-risk) ในกรณีไม่มีปันผลเป็น

$$
\theta=\frac{\mu-r}{\sigma}.
$$

สำหรับค่าคงที่นี้ ให้ density process

$$
Z_t=\exp\left(-\theta W_t^{P}-\frac12\theta^2t\right),\qquad
\left.\frac{dQ}{dP}\right|_{\mathcal F_t}=Z_t.
$$

Z คือ [Radon–Nikodym density](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#radon-nikodym-density) ที่ใช้ถ่วงน้ำหนักผลลัพธ์เดิม สำหรับตัวแปรสุ่ม X ที่เหมาะสม ณ เวลา T:

$$
\mathbb E^{Q}[X]=\mathbb E^{P}[Z_TX].
$$

Z เป็นบวกและมีค่าเฉลี่ยหนึ่ง เหตุการณ์ที่มีโอกาสศูนย์จึงยังมีโอกาสศูนย์เหมือนกันทั้งสองมาตรวัด เราเรียก P กับ Q ว่า **equivalent** แต่น้ำหนักของเหตุการณ์ที่เป็นไปได้เปลี่ยนได้

ทฤษฎีบท Girsanov บอกว่า

$$
W_t^{Q}=W_t^{P}+\theta t
$$

เป็น Brownian motion **ภายใต้ Q** แทน \(dW_t^{P}=dW_t^{Q}-\theta dt\) ในสมการหุ้น:

$$
dS_t=(\mu-\sigma\theta)S_t\,dt+\sigma S_t\,dW_t^{Q}
=rS_t\,dt+\sigma S_t\,dW_t^{Q}.
$$

ดังนั้น \(d\widetilde S_t=\sigma\widetilde S_t\,dW_t^{Q}\) และ discounted GBM นี้เป็น Q-martingale เส้นทางที่อธิบายยังเป็นตัวแปรสุ่ม S เดิม เราเปลี่ยนมาตรวัดที่ใช้เฉลี่ย ไม่ได้ทำให้ผลตอบแทนจริงของหุ้นเปลี่ยนจาก μ เป็น r

### ระวังเครื่องหมายและเงื่อนไข

ในบทนี้ Z ใช้เครื่องหมายลบหน้า θW ส่วน Wᴽ ใช้เครื่องหมายบวกหน้า θt ถ้าเปลี่ยนนิยาม θ เป็น (r−μ)/σ ต้องเปลี่ยนเครื่องหมายทั้งคู่ให้สอดคล้องกัน

ถ้า θ เปลี่ยนตามเวลาและสถานะ จะใช้

$$
Z_t=\exp\left(-\int_0^t\theta_u\,dW_u^{P}
-\frac12\int_0^t\theta_u^2\,du\right).
$$

เงื่อนไข Novikov

$$
\mathbb E^{P}\left[\exp\left(\frac12\int_0^T\theta_u^2\,du\right)\right]<\infty
$$

เป็นเงื่อนไข **เพียงพอ** ให้ Z เป็น true martingale สำหรับ θ คงที่และ T จำกัด ตรวจได้ทันที แต่ Novikov ไม่ใช่เงื่อนไขจำเป็นในทุกกรณี และการเห็นว่า SDE ไม่มี drift เพียงอย่างเดียวยังไม่รับประกัน true martingale ของกระบวนการทั่วไป

<a id="fundamental-pricing"></a>

## จากพอร์ตเลียนแบบสู่ Fundamental Asset Pricing Formula

ให้ \(\widetilde V_t=V_t/B_t\) สำหรับพอร์ต self-financing เมื่อไม่มีปันผล:

$$
d\widetilde V_t=\Delta_t\,d\widetilde S_t
=\Delta_t\sigma\widetilde S_t\,dW_t^{Q}.
$$

สำหรับพอร์ตเลียนแบบที่ stochastic integral นี้เป็น true martingale เช่น มีเงื่อนไข square-integrability ที่เหมาะสม เราใช้ค่าคาดหมายแบบมีเงื่อนไขได้ ถ้าพอร์ตจ่าย H ที่เวลา T จะได้

$$
\frac{V_t}{B_t}=\mathbb E^{Q}\left[\frac{H}{B_T}\mid\mathcal F_t\right],
\qquad
\boxed{V_t=e^{-r(T-t)}\mathbb E^{Q}[H\mid\mathcal F_t]}.
$$

นี่คือราคาพอร์ตที่เลียนแบบ payoff ได้ ถ้า Option ราคาไม่เท่ากับพอร์ต เราซื้อด้านที่ถูกและขายด้านที่แพง ผลตอบแทนปลายทางหักล้างกัน เหลือส่วนต่างที่ขัดกับ no-arbitrage ภายใต้สมมติฐานตลาดที่ใช้

### ทำไมราคานี้มีเพียงค่าเดียวใน Black–Scholes

ในแบบจำลองนี้ Brownian motion มีแหล่งเดียวและหุ้นมี σ≠0 ภายใต้ Brownian filtration และเงื่อนไข integrability ที่เหมาะสม martingale representation ทำให้แปลง discounted payoff เป็นการถือหุ้นกับเงินสดได้ ตลาดจึง [complete](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#complete-market) สำหรับกลุ่ม claims ที่พิจารณา และ Q ของ numeraire นี้มีเพียงมาตรวัดเดียว

ตลาดที่มีแหล่งเสี่ยงซึ่งซื้อขายเพื่อ hedge ไม่ได้อาจมีหลาย Q ที่สอดคล้องกับ no-arbitrage ความไม่มี arbitrage เพียงอย่างเดียวจึงไม่ได้ให้ราคาเดียวสำหรับทุกสัญญาในทุกแบบจำลอง

<a id="measure-experiment"></a>

## ลองเฉลี่ยภายใต้ P, Q และ P ที่ถ่วงน้ำหนัก

ห้องทดลองใช้หุ้นไม่มีปันผล S₀=K=100 และจำลองราคาปลายงวด GBM แบบ exact วิธีแรกเฉลี่ย payoff ภายใต้ P แล้ว discount ด้วย r; วิธีที่สองจำลองภายใต้ Q โดยตรง; วิธีที่สามใช้เส้นทาง P แล้วคูณ Z ก่อนเฉลี่ย

$$
\widehat V_{P\to Q}=\frac1n\sum_{i=1}^{n}
e^{-rT}Z_T^{(i)}(S_T^{P,(i)}-K)^+.
$$

ตัวประมาณนี้หารด้วย n ไม่ใช่ผลรวมน้ำหนัก เพราะ Z มี normalization ตามทฤษฎีอยู่แล้ว ในตัวอย่างจำกัด ค่าเฉลี่ย Z อาจไม่เท่ากับหนึ่งพอดี

ลองเปลี่ยน μ โดยคง r, σ และ T ไว้ **ราคา analytic ภายใต้ Q ต้องไม่เปลี่ยน** แต่ payoff คาดหมายภายใต้ P และน้ำหนัก Z เปลี่ยนได้ จากนั้นปรับ σ และดูว่าราคา Option เปลี่ยนอย่างไร

Standard error (SE) ที่แสดงวัดความคลาดเคลื่อนจากจำนวนตัวอย่าง ค่า Monte Carlo สองวิธีไม่จำเป็นต้องตรงกันพอดี และการเปลี่ยน μ อาจทำให้วิธีถ่วงน้ำหนักมีความแปรปรวนสูงขึ้น แม้คำตอบเชิงทฤษฎีเท่าเดิม การเพิ่มจำนวนตัวอย่างไม่ช่วยแก้ model error

<a id="call-expectation"></a>

## กลับมาถึงสูตร Call ผ่าน Lognormal

ภายใต้ Q และเมื่อทราบ Sₜ ราคาปลายงวดเขียนเป็น

$$
S_T=S_t\exp\left[\left(r-\frac12\sigma^2\right)\tau
+\sigma\sqrt\tau\,\xi\right],\qquad \xi\sim N(0,1).
$$

กำหนด \(\Phi\) เป็น CDF ของ Standard Normal และ \(\varphi\) เป็น density ตั้ง

$$
d_2=\frac{\log(S_t/K)+(r-\tfrac12\sigma^2)\tau}{\sigma\sqrt\tau},
\qquad d_1=d_2+\sigma\sqrt\tau.
$$

Call มี payoff เมื่อ ξ>−d₂ เราจึงแยกค่าเฉลี่ยได้เป็น

$$
C_t=e^{-r\tau}\mathbb E^Q[S_T\mathbf1_{\{S_T>K\}}\mid\mathcal F_t]
-Ke^{-r\tau}Q(S_T>K\mid\mathcal F_t).
$$

พจน์ที่สองให้ \(Ke^{-r\tau}\Phi(d_2)\) ส่วนพจน์แรกใช้การ complete the square:

$$
e^{-\sigma^2\tau/2+\sigma\sqrt\tau z}\varphi(z)
=\varphi(z-\sigma\sqrt\tau).
$$

เมื่อเลื่อนขอบเขตอินทิกรัลจาก −d₂ ไปเป็น −d₁ จะได้

$$
\boxed{C_t=S_t\Phi(d_1)-Ke^{-r\tau}\Phi(d_2)}.
$$

Put ได้จาก put–call parity:

$$
P_t=Ke^{-r\tau}\Phi(-d_2)-S_t\Phi(-d_1),\qquad
C_t-P_t=S_t-Ke^{-r\tau}.
$$

สำหรับ S=K=100, r=5%, σ=20% และ τ=1 ปี ได้ d₁=0.35, d₂=0.15, Call≈10.4506 และ Put≈5.5735 ต่อหนึ่งหน่วยหุ้น สอดคล้องกับบท Black–Scholes เดิม

<a id="numeraire"></a>

## ทำไม Φ(d₁) กับ Φ(d₂) จึงเป็นคนละความน่าจะเป็น

จากนิยามข้างต้น \(\Phi(d_2)=Q(S_T>K\mid\mathcal F_t)\) แต่ \(\Phi(d_1)\) ไม่ใช่ความน่าจะเป็นเดียวกันภายใต้ Q

เลือกหุ้นไม่มีปันผลเป็น [numeraire](https://nutdnuy.github.io/quantitative-finance-notes/glossary.html#numeraire) แทนบัญชีเงินสด เราจะได้มาตรวัด \(Q^S\) ที่มี density เทียบกับ Q:

$$
\left.\frac{dQ^S}{dQ}\right|_{\mathcal F_t}
=\frac{S_t/B_t}{S_0/B_0}.
$$

เส้นทางที่หุ้นมีมูลค่าสูงได้รับน้ำหนักมากขึ้นภายใต้มาตรวัดนี้ และ

$$
Q^S(S_T>K\mid\mathcal F_t)=\Phi(d_1).
$$

Call จึงเขียนได้เป็น \(S_t Q^S(S_T>K\mid\mathcal F_t)-Ke^{-r\tau}Q(S_T>K\mid\mathcal F_t)\) สองพจน์ใช้น้ำหนักคนละมาตรวัด แต่รวมกันเป็นราคาในหน่วยเงินเดียวกัน

| สัญญา | Payoff | ราคาวันนี้ เมื่อไม่มีปันผล |
|---|---|---|
| Cash-or-nothing Call จ่ายเงิน 1 | \(\mathbf1_{\{S_T>K\}}\) | \(e^{-r\tau}\Phi(d_2)\) |
| Asset-or-nothing Call จ่ายหุ้น 1 หน่วย | \(S_T\mathbf1_{\{S_T>K\}}\) | \(S_t\Phi(d_1)\) |
| Vanilla Call | \((S_T-K)^+\) | Asset binary − K × Cash binary |

ตัวอย่างเดิมให้ Φ(d₂)≈0.5596 และ Φ(d₁)≈0.6368 ราคา asset binary≈63.6831 ส่วน K เท่าของ cash binary≈53.2325 ผลต่างคือ Call≈10.4506 การนำ Φ(d₁) ไปเรียกว่าโอกาสใช้สิทธิภายใต้ Q หรือ P โดยไม่ระบุมาตรวัดจึงคลาดเคลื่อน

สูตรทั่วไปสำหรับ numeraire N คือ

$$
V_t=N_t\mathbb E^{Q^N}\left[\frac{H}{N_T}\mid\mathcal F_t\right].
$$

N ต้องเป็นสินทรัพย์หรือพอร์ต self-financing ที่ซื้อขายได้และเป็นบวกอย่างเคร่งครัด พร้อมเงื่อนไขให้การเปลี่ยนมาตรวัดถูกต้อง ไม่ใช่เลือกกระบวนการบวกใด ๆ มาหารราคาแล้วอ้างสูตรนี้ได้ทันที

<a id="feynman-kac"></a>

## Feynman–Kac เชื่อมค่าเฉลี่ยกับ PDE

ให้ X ภายใต้ **มาตรวัดเดียวกับที่ใช้คาดหมาย** มี dynamics \(dX_u=a(u,X_u)du+b(u,X_u)dW_u\) ภายใต้เงื่อนไขความเรียบ การเติบโต และ integrability ที่เหมาะสม ฟังก์ชัน

$$
v(t,x)=\mathbb E\left[
e^{-\int_t^T c(u,X_u)du}g(X_T)\mid X_t=x\right]
$$

สัมพันธ์กับสมการ

$$
v_t+a(t,x)v_x+\frac12b(t,x)^2v_{xx}-c(t,x)v=0,
\qquad v(T,x)=g(x).
$$

ใน Black–Scholes ภายใต้ Q ใช้ a=rS, b=σS, c=r จึงได้

$$
V_t+rSV_S+\frac12\sigma^2S^2V_{SS}-rV=0,
\qquad V(T,S)=G(S).
$$

สมการตรงกับที่ได้จาก Delta hedge แต่ **Feynman–Kac ไม่ได้เลือก Q ให้เรา** ถ้าใส่ drift μ ภายใต้ P จะได้สมการสำหรับค่าคาดหมายภายใต้ P การได้ pricing measure ต้องมาจากเงื่อนไขตลาดและ no-arbitrage ก่อน

วิธีความน่าจะเป็นยังให้ข้อมูล hedge ได้ เมื่อหาฟังก์ชัน V แล้วใช้ Itô เปรียบเทียบส่วน Brownian จะได้ \(\Delta=V_S\) เหมือนเดิม วิธี PDE, Monte Carlo และสูตร analytic จึงเป็นเครื่องมือคำนวณที่เชื่อมกันภายใต้แบบจำลองเดียวกัน

<a id="dividends"></a>

## หุ้นจ่ายปันผล: ต้องนับผลตอบแทนรวม

ให้ D เป็น **continuous dividend yield ต่อปี** ไม่ใช่เงินปันผลก้อนคงที่ และใช้ D แทน q เพื่อไม่ให้สับสนกับน้ำหนัก Binomial ในส่วนนี้กำหนด μ เป็น expected **total return** ภายใต้ P:

$$
dS_t=(\mu-D)S_t\,dt+\sigma S_t\,dW_t^P,
\qquad d\text{Gain}_t=dS_t+DS_t\,dt.
$$

ภายใต้ Q drift ของราคาหุ้นที่ไม่รวมปันผลเป็น r−D:

$$
dS_t=(r-D)S_t\,dt+\sigma S_t\,dW_t^Q.
$$

ตอนนี้ S/B เพียงอย่างเดียวไม่ใช่ Q-martingale เมื่อ D≠0 สิ่งที่เป็น martingale คือ discounted gains ซึ่งรวมปันผล:

$$
\frac{S_t}{B_t}+\int_0^t\frac{DS_u}{B_u}\,du.
$$

หรือใช้มูลค่าพอร์ตที่นำปันผลกลับไปลงทุนในหุ้น \(S_t^{\mathrm{TR}}=e^{Dt}S_t\) แล้วหารด้วย Bₜ เมื่อ D คงที่ จึงต้องระวังว่ากำลังใช้ ex-dividend price หรือ total-return asset เป็น numeraire

สูตรราคากลายเป็น

$$
C_t=S_te^{-D\tau}\Phi(d_1)-Ke^{-r\tau}\Phi(d_2),
$$

$$
d_1=\frac{\log(S_t/K)+(r-D+\tfrac12\sigma^2)\tau}{\sigma\sqrt\tau},
\qquad d_2=d_1-\sigma\sqrt\tau.
$$

ตัวอย่างเดิมเพิ่ม D=2% ได้ Call≈9.2270 และ Put≈6.3301 โดย \(C-P=Se^{-D\tau}-Ke^{-r\tau}\) สมมติฐาน continuous yield นี้ไม่ใช่แบบจำลองปันผลเงินสดก้อนที่จ่ายตามวัน ex-dividend

<a id="time-dependent-parameters"></a>

## พารามิเตอร์เปลี่ยนตามเวลา: รวม variance ก่อนถอดราก

ถ้า r(u), D(u), σ(u) เป็นฟังก์ชันเวลา **ที่ทราบแน่นอน** ให้

$$
R=\int_t^T r(u)\,du,\qquad
Y=\int_t^T D(u)\,du,\qquad
A=\int_t^T\sigma(u)^2\,du.
$$

Y คือ yield สะสม ส่วน A คือความแปรปรวนสะสมของ log return ไม่ใช่ volatility เฉลี่ย ภายใต้ Q:

$$
\log(S_T/S_t)\sim N\left(R-Y-\frac12A,\ A\right).
$$

จึงได้

$$
C_t=S_te^{-Y}\Phi(d_1)-Ke^{-R}\Phi(d_2),
\qquad d_1=\frac{\log(S_t/K)+R-Y+A/2}{\sqrt A},
\quad d_2=d_1-\sqrt A.
$$

สูตรนี้เขียนสำหรับ A>0 ถ้า A=0 payoff ไม่สุ่มภายใต้ Q และราคา Call คือ \(\max(S_te^{-Y}-Ke^{-R},0)\)

ตัวอย่างครึ่งปีแรก r=4%, D=1%, σ=10% และครึ่งปีหลัง r=6%, D=3%, σ=30%:

$$
R=0.05,\qquad Y=0.02,\qquad
A=0.5(0.10)^2+0.5(0.30)^2=0.05.
$$

Volatility เทียบเท่าหนึ่งปีคือ \(\sqrt{A/1}\approx22.3607\%\) ไม่ใช่ค่าเฉลี่ยเลขคณิต 20% เพราะสิ่งที่บวกข้ามเวลาคือ variance ตัวอย่างนี้คำนวณราคาและตรวจ parity ได้ใน Notebook

ถ้า σ หรือ r เป็นตัวแปรสุ่ม สูตรที่แทนด้วยอินทิกรัล deterministic นี้ใช้ไม่ได้โดยอัตโนมัติ โดยเฉพาะดอกเบี้ยสุ่ม ตัว discount factor ต้องอยู่ในค่าคาดหมายตามกรอบที่ใช้

<a id="black-76"></a>

## Black–76: Option บนราคา Futures

แยกเวลาสองตัวให้ชัด: T คือวันหมดอายุ **Option** และ U≥T คือวันครบกำหนด **Futures** ที่ Option อ้างอิง ให้ \(F_t=F(t,U)\) เป็นราคา Futures ณ เวลา t สัญญา European Call จ่าย \((F_T-K)^+\) ณ เวลา T

ในแบบจำลองอัตราดอกเบี้ย deterministic และหุ้นที่ใช้ cost-of-carry ได้ ราคา forward กับ futures ตรงกัน ถ้าไม่มีปันผลและ r คงที่จะได้ \(F(t,U)=S_te^{r(U-t)}\) หากมีปันผลต่อเนื่องคงที่ ใช้ r−D แทน r

ภายใต้ Q สมมติ Futures เป็น lognormal:

$$
dF_t=\sigma_F F_t\,dW_t^Q.
$$

Futures price ไม่มี drift rF ในสมการนี้ **F เป็นราคาอ้างอิง ไม่ใช่มูลค่าทั้งก้อนที่จ่ายเพื่อซื้อสัญญา Futures** กำไรขาดทุนจาก Futures เข้าบัญชีผ่าน settlement การตั้งพอร์ตต้องรวมเงินสดและ margin cash flows อย่างถูกต้อง

สำหรับสูตรถัดไปให้ r และ σF คงที่ พิจารณา European Option แบบชำระ premium ล่วงหน้าและจ่าย payoff ที่ T:

$$
C_t=e^{-r\tau}\left[F_t\Phi(d_1)-K\Phi(d_2)\right],
$$

$$
P_t=e^{-r\tau}\left[K\Phi(-d_2)-F_t\Phi(-d_1)\right],
$$

$$
d_1=\frac{\log(F_t/K)+\tfrac12\sigma_F^2\tau}{\sigma_F\sqrt\tau},
\qquad d_2=d_1-\sigma_F\sqrt\tau.
$$

เรียกว่า **Black–76** เวลาในสูตรคือ τ=T−t ส่วน U มีผลต่อราคา Futures และ volatility ของสัญญาที่เลือก ไม่ได้นำ U−t มาแทนเวลาหมดอายุ Option

ถ้า F=K=100, r=5%, σF=20%, τ=1 ราคา Call และ Put เท่ากันประมาณ **7.5771** และ parity คือ \(C-P=e^{-r\tau}(F-K)\) สมมติฐานนี้ไม่ครอบคลุม Futures ราคาติดลบหรือรูปแบบ Option ที่ settle ต่างออกไป ส่วนดอกเบี้ยสุ่มอาจทำให้ราคา futures ต่างจาก forward

ลองสลับจาก Spot เป็น Futures โดยคงราคาอ้างอิง 100, strike 100 และพารามิเตอร์อื่นไว้ ค่า 100 ในสองโหมดเป็นคนละปริมาณ จึงไม่ควรคาดว่าราคาจะเท่ากัน ถ้าต้องการตรวจความเทียบเท่ากรณีส่งมอบที่วันหมดอายุ Option ให้ตั้ง \(F=Se^{(r-D)\tau}\) แล้วเทียบสูตรภายใต้สมมติฐานเดียวกัน

<a id="check-understanding"></a>

## ลองตรวจความเข้าใจ

1. ถ้าเพิ่ม μ จาก 8% เป็น 12% โดยคง S, K, r, σ และ T ไว้ ราคา Black–Scholes เปลี่ยนหรือไม่? แล้ว payoff คาดหมายภายใต้ P ล่ะ?
2. S/B เป็น martingale ภายใต้ Q เมื่อหุ้นจ่าย continuous yield D=2% หรือไม่? ต้องเพิ่มอะไรเข้าไป?
3. Φ(d₁) และ Φ(d₂) ต่างกันเพราะอะไร? ทั้งสองเป็นโอกาสที่หุ้นขึ้นภายใต้ P หรือไม่?
4. σ เท่ากับ 10% ครึ่งปีและ 30% อีกครึ่งปี ให้ volatility หนึ่งปีเท่าไร?
5. Option หมดอายุใน 3 เดือน อ้างอิง Futures ครบกำหนดใน 9 เดือน ใช้ τ เท่าไรใน Black–76?

**เปิดแนวคำตอบ**

1. ราคา Q ไม่เปลี่ยน ส่วน payoff คาดหมายของ Call ภายใต้ P เพิ่มใน GBM นี้เมื่อ μ เพิ่ม
2. ไม่ใช่ ต้องใช้ discounted total gains หรือพอร์ตที่นำปันผลกลับมาลงทุนแล้ว discount
3. Φ(d₂) เป็นโอกาสจบ in-the-money ภายใต้ cash-account measure Q ส่วน Φ(d₁) ใช้ stock measure Qˢ ในกรณีไม่มีปันผล ทั้งคู่ไม่ใช่ความน่าจะเป็นจริง P โดยทั่วไป
4. \(\sqrt{0.5(0.1)^2+0.5(0.3)^2}=0.223607\) หรือประมาณ 22.3607% ต่อรากปี
5. τ=0.25 ปี ใช้ราคา Futures ของสัญญาอายุ 9 เดือนเป็น F ปัจจุบัน ส่วน volatility ต้องตรงกับ Futures นั้นและช่วง 3 เดือนของ Option

<a id="sources-and-notebook"></a>

## ทดลองต่อและแหล่งอ่านประกอบ

[ดาวน์โหลด Python Notebook](https://nutdnuy.github.io/quantitative-finance-notes/notebooks/martingale-pricing.ipynb) เพื่อรันการเปลี่ยนมาตรวัด, Monte Carlo สองวิธี, cash/asset binary, พารามิเตอร์สองช่วง และ Black–76 ตัวอย่างใช้ seed ที่ระบุไว้และ Python standard library ผลสุ่มของ Python กับ JavaScript ไม่จำเป็นต้องตรงกัน แต่ต้องสอดคล้องกับสูตรและขอบเขต sampling error

บทนี้เรียบเรียงใหม่จากเอกสารประกอบการเรียน *Martingales Theory: Application to Option Pricing — Black-Scholes All Over Again* ของ CQF ที่ผู้ใช้ให้มา พร้อมตรวจสมการและคำนวณตัวอย่างใหม่ รายละเอียดแหล่งที่มา ขอบเขตหน้า และจุดที่ปรับแก้เก็บใน [บันทึกที่มา](https://nutdnuy.github.io/quantitative-finance-notes/data/martingale-pricing-provenance.json) โดยไม่เผยแพร่ PDF หรือภาพหน้าต้นฉบับ

อ่านกรอบ risk-neutral valuation และปันผลเพิ่มใน [Martin Haugh — The Black-Scholes Model](https://www.columbia.edu/~mh2078/FoundationsFE/BlackScholes.pdf) และศึกษาทฤษฎีการเปลี่ยนมาตรวัดใน [Gregory F. Lawler — Stochastic Calculus: An Introduction with Applications](https://www.math.uchicago.edu/~lawler/finbook.pdf) หัวข้อ Girsanov’s Theorem สูตรในบทขึ้นกับสมมติฐานการเลียนแบบและแบบจำลอง ไม่ใช่ข้อยืนยันว่าความเสี่ยงทุกชนิดในตลาดจริงป้องกันได้หมด

## 1. เตรียมตัวคำนวณและหน่วย

ทุกตัวเลขต่อไปนี้เป็นตัวอย่างสมมติ ราคามีหน่วยเงินต่อสินทรัพย์หนึ่งหน่วย เวลาเป็นปี อัตราดอกเบี้ยและ dividend yield เป็นอัตราทบต้นต่อเนื่องต่อปี ส่วน volatility เป็นค่าต่อรากปี โค้ดใช้ Python standard library เท่านั้น

ฟังก์ชันหลักรับค่าที่สะสมตลอดอายุ Option: \(R=\int r(u)du\), \(Y=\int D(u)du\) และ \(A=\int\sigma(u)^2du\) ใช้ `Y` สำหรับ dividend yield สะสมตามสัญลักษณ์ในบท และใช้ `dividend` สำหรับ dividend yield ต่อปีในฟังก์ชันค่าคงที่

In [1]:
import math
import random
import statistics

def close(actual, expected, tolerance=1e-10):
    assert math.isclose(actual, expected, rel_tol=tolerance, abs_tol=tolerance), (actual, expected)

def normal_cdf(x):
    return 0.5 * math.erfc(-x / math.sqrt(2.0))

def normal_pdf(x):
    return math.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)

def generalized_bs(S, K, R, Y, A):
    """European prices for deterministic rates, dividend yield and volatility.

    R and Y are integrated rates; A is integrated variance, not volatility.
    """
    if not all(math.isfinite(x) for x in (S, K, R, Y, A)):
        raise ValueError("Inputs must be finite")
    if S <= 0 or K <= 0 or A < 0:
        raise ValueError("Require S,K > 0 and integrated variance A >= 0")
    stock_pv, strike_pv = S * math.exp(-Y), K * math.exp(-R)
    if A == 0:
        return dict(call=max(stock_pv-strike_pv, 0.0),
                    put=max(strike_pv-stock_pv, 0.0), d1=None, d2=None)
    d1 = (math.log(S/K) + R - Y + 0.5*A) / math.sqrt(A)
    d2 = d1 - math.sqrt(A)
    return dict(call=stock_pv*normal_cdf(d1)-strike_pv*normal_cdf(d2),
                put=strike_pv*normal_cdf(-d2)-stock_pv*normal_cdf(-d1),
                d1=d1, d2=d2)

def black_scholes(S, K, r, dividend, sigma, tau):
    if sigma < 0 or tau < 0:
        raise ValueError("Require sigma,tau >= 0")
    return generalized_bs(S, K, r*tau, dividend*tau, sigma*sigma*tau)

def estimate(values):
    if len(values) < 2:
        raise ValueError("At least two samples are required for sample standard error")
    return statistics.fmean(values), statistics.stdev(values) / math.sqrt(len(values))

def report(label, values, target):
    mean, se = estimate(values)
    print(f"{label}: estimate={mean:.6f}, SE={se:.6f}, target={target:.6f}")
    print(f"  Approximate 95% sampling interval [{mean-1.96*se:.6f}, {mean+1.96*se:.6f}]")
    return mean, se

S0, K, mu, r, sigma, T = 100.0, 100.0, 0.12, 0.05, 0.20, 1.0
benchmark = black_scholes(S0, K, r, 0.0, sigma, T)
close(benchmark["call"], 10.450583572185565)
close(benchmark["put"], 5.573526022256971)
close(benchmark["call"]-benchmark["put"], S0-K*math.exp(-r*T))
close(black_scholes(100, 90, .05, 0, .2, 0)["call"], 10)
close(black_scholes(100, 100, .05, 0, 0, 1)["call"], 100-100*math.exp(-.05))
print(f"Analytical benchmark: Call={benchmark['call']:.9f}; Put={benchmark['put']:.9f}")
print("All examples are hypothetical; the notebook has no data downloads or third-party package requirements.")

Analytical benchmark: Call=10.450583572; Put=5.573526022
All examples are hypothetical; the notebook has no data downloads or third-party package requirements.


## 2. เปลี่ยนจาก P เป็น Q ด้วยน้ำหนัก Radon–Nikodym

เริ่มด้วยหุ้นไม่มีปันผล \(dS_t=\mu S_tdt+\sigma S_tdW_t^P\) กำหนด \(\theta=(\mu-r)/\sigma\) และ \(Z_T=\exp(-\theta W_T^P-\tfrac12\theta^2T)\) โดย \(dQ/dP=Z_T\) การคิดค่าเฉลี่ยภายใต้ P แล้วคูณน้ำหนักนี้ให้ \(E_P[Z_TX]=E_Q[X]\)

ตรวจทั้ง \(E_P[Z_T]=1\), \(E_P[Z_Te^{-rT}S_T]=S_0\) และราคา Call แล้วเทียบกับการสุ่ม GBM ภายใต้ Q โดยตรง ตัวอย่างใช้สอง seed แยกกัน และใช้ค่าเฉลี่ย `sum(Z * discounted_payoff) / n` โดยไม่หารด้วยผลรวมของ `Z` การหารด้วยผลรวมของน้ำหนักจะเปลี่ยนเป็นอีก estimator หนึ่ง

ช่วงที่รายงานเป็น normal approximation ของ sampling uncertainty ภายใต้โมเดล ไม่ได้ครอบคลุม model error และไม่ได้บังคับว่าทุกรอบต้องครอบคลุมราคาสูตร

In [2]:
theta = (mu-r) / sigma
paths, seed_p, seed_q = 100000, 2026091901, 2026091902
discount = math.exp(-r*T)
rng_p, rng_q = random.Random(seed_p), random.Random(seed_q)
weights, weighted_stock, weighted_call, naive_p_call = [], [], [], []
q_terminal, direct_q_call = [], []
max_pathwise_residual = 0.0
for _ in range(paths):
    Wp = math.sqrt(T) * rng_p.gauss(0.0, 1.0)
    Sp = S0 * math.exp((mu-0.5*sigma*sigma)*T + sigma*Wp)
    Z = math.exp(-theta*Wp - 0.5*theta*theta*T)
    weights.append(Z)
    weighted_stock.append(Z*discount*Sp)
    weighted_call.append(Z*discount*max(Sp-K, 0.0))
    naive_p_call.append(discount*max(Sp-K, 0.0))
    # This identity holds on every sampled P path, not only in expectation.
    rhs = S0*math.exp((sigma-theta)*Wp - 0.5*(sigma-theta)**2*T)
    max_pathwise_residual = max(max_pathwise_residual, abs(Z*discount*Sp-rhs))
    Wq = math.sqrt(T) * rng_q.gauss(0.0, 1.0)
    Sq = S0 * math.exp((r-0.5*sigma*sigma)*T + sigma*Wq)
    q_terminal.append(Sq)
    direct_q_call.append(discount*max(Sq-K, 0.0))

# Gaussian exponential moments establish these identities analytically.
close(math.exp(-0.5*theta*theta*T)*math.exp(0.5*theta*theta*T), 1.0)
close(S0*math.exp((mu-r-0.5*sigma*sigma-0.5*theta*theta)*T
                 +0.5*(sigma-theta)**2*T), S0)
assert max_pathwise_residual < 1e-10
assert all(z > 0 and math.isfinite(z) for z in weights)
print(f"theta={theta:.6f}; paths={paths:,}; P seed={seed_p}; Q seed={seed_q}; dividend yield=0")
report("E_P[Z_T]", weights, 1.0)
report("E_P[Z_T exp(-rT) S_T]", weighted_stock, S0)
p_price, p_se = report("Call via weighted P", weighted_call, benchmark["call"])
q_price, q_se = report("Call via direct Q", direct_q_call, benchmark["call"])
naive_mean, naive_se = estimate(naive_p_call)
print(f"Discounted physical expected payoff without Z: {naive_mean:.6f}, SE={naive_se:.6f}")
print("The last estimate is a physical expectation, not this model's no-arbitrage call premium.")
print(f"Weighted-P minus direct-Q={p_price-q_price:+.6f}; independent-draw SE={math.hypot(p_se,q_se):.6f}")
print(f"Maximum pathwise identity residual={max_pathwise_residual:.3e}")
print("Changing measure changes scenario weights; it does not remove volatility or make risky cash flows certain.")

theta=0.350000; paths=100,000; P seed=2026091901; Q seed=2026091902; dividend yield=0
E_P[Z_T]: estimate=1.001608, SE=0.001143, target=1.000000
  Approximate 95% sampling interval [0.999368, 1.003847]
E_P[Z_T exp(-rT) S_T]: estimate=100.069546, SE=0.047726, target=100.000000
  Approximate 95% sampling interval [99.976003, 100.163090]
Call via weighted P: estimate=10.412062, SE=0.030516, target=10.450584
  Approximate 95% sampling interval [10.352250, 10.471873]
Call via direct Q: estimate=10.505157, SE=0.046734, target=10.450584
  Approximate 95% sampling interval [10.413557, 10.596756]
Discounted physical expected payoff without Z: 15.455764, SE=0.056196
The last estimate is a physical expectation, not this model's no-arbitrage call premium.
Weighted-P minus direct-Q=-0.093095; independent-draw SE=0.055815
Maximum pathwise identity residual=8.527e-14
Changing measure changes scenario weights; it does not remove volatility or make risky cash flows certain.


## 3. แยก Call เป็นขาสินทรัพย์กับขาเงินสด

สมการ payoff คือ \((S_T-K)^+=S_T1_{S_T>K}-K1_{S_T>K}\) ดังนั้นราคา Call เท่ากับราคา asset-or-nothing ลบ \(K\) เท่าของราคา cash-or-nothing ที่จ่ายเงินหนึ่งหน่วย

ในตัวอย่างไม่มีปันผล \(N(d_2)=Q(S_T>K)\) แต่ \(N(d_1)=Q^S(S_T>K)\) เมื่อเปลี่ยน numeraire จากบัญชีเงินสดมาเป็นหุ้น โดย \(dQ^S/dQ=e^{-rT}S_T/S_0\) สองค่านี้ใช้เหตุการณ์ exercise เดียวกัน แต่คนละ measure; ทั้งคู่ไม่ได้เป็นความน่าจะเป็นจริงภายใต้ P โดยอัตโนมัติ

In [3]:
d1, d2 = benchmark["d1"], benchmark["d2"]
q_exercise, stock_measure_exercise = normal_cdf(d2), normal_cdf(d1)
cash_binary = discount*q_exercise  # Pays one currency unit on exercise.
asset_binary = S0*stock_measure_exercise  # Pays one stock unit on exercise.
close(asset_binary-K*cash_binary, benchmark["call"])
stock_measure_density = [discount*s/S0 for s in q_terminal]
q_indicators = [float(s > K) for s in q_terminal]
stock_weighted_indicators = [z*event for z, event in zip(stock_measure_density, q_indicators)]
print(f"d1={d1:.6f}; d2={d2:.6f}")
print(f"Q exercise probability N(d2)={q_exercise:.9f}")
print(f"Stock-numeraire exercise probability N(d1)={stock_measure_exercise:.9f}")
print(f"One-unit cash binary={cash_binary:.9f}; asset binary={asset_binary:.9f}")
print(f"Call = {asset_binary:.9f} - {K:.0f} * {cash_binary:.9f} = {benchmark['call']:.9f}")
report("E_Q[dQ^S/dQ]", stock_measure_density, 1.0)
report("Q exercise probability", q_indicators, q_exercise)
report("Q^S exercise probability via weighted Q", stock_weighted_indicators, stock_measure_exercise)
print("No sample weight normalization is used; weighted sample estimates need not lie in [0,1] in every finite sample.")
print("With dividends, use the reinvested total-return stock as numeraire, not the ex-dividend stock alone.")

d1=0.350000; d2=0.150000
Q exercise probability N(d2)=0.559617692
Stock-numeraire exercise probability N(d1)=0.636830651
One-unit cash binary=0.532324815; asset binary=63.683065118
Call = 63.683065118 - 100 * 0.532324815 = 10.450583572
E_Q[dQ^S/dQ]: estimate=1.000548, SE=0.000641, target=1.000000
  Approximate 95% sampling interval [0.999292, 1.001803]
Q exercise probability: estimate=0.559370, SE=0.001570, target=0.559618
  Approximate 95% sampling interval [0.556293, 0.562447]
Q^S exercise probability via weighted Q: estimate=0.637141, SE=0.001825, target=0.636831
  Approximate 95% sampling interval [0.633564, 0.640717]
No sample weight normalization is used; weighted sample estimates need not lie in [0,1] in every finite sample.
With dividends, use the reinvested total-return stock as numeraire, not the ex-dividend stock alone.


## 4. Feynman–Kac: ตรวจความคาดหมายกับ PDE

สูตรราคาที่ได้จาก discounted expectation ภายใต้ Q ต้องสอดคล้องกับ Black–Scholes PDE ภายใต้สมมติฐานเดียวกัน เซลล์นี้ใช้ central differences ตรวจอนุพันธ์อย่างอิสระจากสูตร Greeks โดยใช้ \(V_t=-V_\tau\) เพราะเวลาคงเหลือ \(\tau=T-t\) ลดลงเมื่อเวลาปฏิทินเดินหน้า

Feynman–Kac เชื่อม PDE กับ expectation; การเลือก Q สำหรับราคาเกิดจาก เงื่อนไข no-arbitrage และการซื้อขายเลียนแบบ payoff ไม่ใช่จากทฤษฎีนี้เพียงอย่างเดียว

In [4]:
dividend = 0.02
def call_value(stock=S0, tau=T):
    return black_scholes(stock, K, r, dividend, sigma, tau)["call"]

h_s, h_t = 0.01, 1e-4
value = call_value()
delta_fd = (call_value(S0+h_s)-call_value(S0-h_s))/(2*h_s)
gamma_fd = (call_value(S0+h_s)-2*value+call_value(S0-h_s))/(h_s*h_s)
calendar_theta_fd = -(call_value(tau=T+h_t)-call_value(tau=T-h_t))/(2*h_t)
pde_residual = (calendar_theta_fd + 0.5*sigma*sigma*S0*S0*gamma_fd
                + (r-dividend)*S0*delta_fd - r*value)
assert abs(pde_residual) < 2e-6, pde_residual
print(f"With dividend yield={dividend:.2%}: Call={value:.9f}")
print(f"Finite-difference Delta={delta_fd:.9f}; Gamma={gamma_fd:.9f}; Theta/year={calendar_theta_fd:.9f}")
print(f"PDE residual={pde_residual:.3e}; tolerance=2e-6 currency units per year.")
print("This numerical residual checks formula consistency; it is not a proof of the theorem or a market-model validation.")

With dividend yield=2.00%: Call=9.227005508
Finite-difference Delta=0.586851139; Gamma=0.018950579; Theta/year=-5.089318919
PDE residual=-5.028e-08; tolerance=2e-6 currency units per year.
This numerical residual checks formula consistency; it is not a proof of the theorem or a market-model validation.


## 5. ดอกเบี้ย ปันผล และ volatility ที่เปลี่ยนตามเวลา

แบ่งปีเป็นสองช่วง ช่วงละ 0.5 ปี ใช้ \((r,D,\sigma)\) เท่ากับ \((4\%,1\%,10\%)\) และ \((6\%,3\%,30\%)\) ตามลำดับ ค่าทั้งหมดเป็น deterministic จึงได้ \(R=0.05\), \(Y=0.02\), \(A=0.05\) และ effective volatility \(\sqrt{A/T}\approx22.36\%\) ต้องเฉลี่ย **variance** ตามเวลา ไม่ใช่เฉลี่ย volatility แล้วนำไปยกกำลังสอง

ตรวจสูตรด้วยการอินทิเกรต payoff บนความหนาแน่น Normal โดยตรง วิธี Simpson นี้ไม่ได้เรียก CDF ของสูตรราคา จึงช่วยตรวจการใช้ discount และ drift อีกทางหนึ่ง รวมทั้งตรวจ put–call parity และการลดรูปเมื่อค่าคงที่

In [5]:
# Tuple fields: duration in years, short rate, dividend yield, volatility.
segments = [(0.5, 0.04, 0.01, 0.10), (0.5, 0.06, 0.03, 0.30)]
term_T = math.fsum(dt for dt, _, _, _ in segments)
R = math.fsum(dt*rate for dt, rate, _, _ in segments)
Y = math.fsum(dt*yield_ for dt, _, yield_, _ in segments)
A = math.fsum(dt*vol*vol for dt, _, _, vol in segments)
close(R, .05)
close(Y, .02)
close(A, .05)
term_price = generalized_bs(S0, K, R, Y, A)
close(term_price["call"]-term_price["put"], S0*math.exp(-Y)-K*math.exp(-R))
effective = black_scholes(S0, K, R/term_T, Y/term_T, math.sqrt(A/term_T), term_T)
close(effective["call"], term_price["call"])
close(effective["put"], term_price["put"])
constant = generalized_bs(S0, K, r*T, 0.0, sigma*sigma*T)
close(constant["call"], benchmark["call"])

def simpson(function, left, right, panels=12000):
    if panels <= 0 or panels % 2 or right < left:
        raise ValueError("Require even positive panels and ordered integration bounds")
    if left == right:
        return 0.0
    h = (right-left)/panels
    total = function(left)+function(right)
    total += 4*math.fsum(function(left+i*h) for i in range(1, panels, 2))
    total += 2*math.fsum(function(left+i*h) for i in range(2, panels, 2))
    return h*total/3

def payoff_quadrature(S, strike, rate_integral, dividend_integral, variance_integral):
    if variance_integral <= 0:
        raise ValueError("This quadrature example requires strictly positive integrated variance")
    root_A = math.sqrt(variance_integral)
    cutoff = (math.log(strike/S)-rate_integral+dividend_integral+0.5*variance_integral)/root_A
    bound = 12.0
    split = min(bound, max(-bound, cutoff))
    def terminal(z):
        return S*math.exp(rate_integral-dividend_integral-0.5*variance_integral+root_A*z)
    call_integral = simpson(lambda z: max(terminal(z)-strike, 0.0)*normal_pdf(z), split, bound)
    put_integral = simpson(lambda z: max(strike-terminal(z), 0.0)*normal_pdf(z), -bound, split)
    return math.exp(-rate_integral)*call_integral, math.exp(-rate_integral)*put_integral

quad_call, quad_put = payoff_quadrature(S0, K, R, Y, A)
close(quad_call, term_price["call"], 1e-8)
close(quad_put, term_price["put"], 1e-8)
average_vol = math.fsum(dt*vol for dt, _, _, vol in segments)/term_T
wrong_variance_price = generalized_bs(S0, K, R, Y, average_vol**2*term_T)["call"]
print(f"R={R:.6f}; Y={Y:.6f}; integrated variance A={A:.6f}")
print(f"Effective volatility sqrt(A/T)={math.sqrt(A/term_T):.6%}; arithmetic average volatility={average_vol:.6%}")
print(f"Generalized formula: Call={term_price['call']:.9f}; Put={term_price['put']:.9f}")
print(f"Independent quadrature: Call={quad_call:.9f}; Put={quad_put:.9f}")
print(f"Formula-minus-quadrature errors: {term_price['call']-quad_call:+.3e}, {term_price['put']-quad_put:+.3e}")
print(f"Call using the incorrect arithmetic-volatility average: {wrong_variance_price:.9f}")
print("Quadrature truncates standard Normal shocks to [-12,12]; tails are negligible for these example parameters.")
print("These integrated-parameter formulas do not extend automatically to stochastic rates or stochastic volatility.")

R=0.050000; Y=0.020000; integrated variance A=0.050000
Effective volatility sqrt(A/T)=22.360680%; arithmetic average volatility=20.000000%
Generalized formula: Call=10.122244497; Put=7.225319617
Independent quadrature: Call=10.122244497; Put=7.225319617
Formula-minus-quadrature errors: -1.474e-13, -1.306e-13
Call using the incorrect arithmetic-volatility average: 9.227005508
Quadrature truncates standard Normal shocks to [-12,12]; tails are negligible for these example parameters.
These integrated-parameter formulas do not extend automatically to stochastic rates or stochastic volatility.


## 6. Black 76 และวันที่สองวัน

กำหนดราคาอ้างอิง forward/futures ปัจจุบัน \(F_0=100\), strike 100, ดอกเบี้ย 5%, volatility ของราคาอ้างอิง 20%, Option หมดอายุใน \(T=1\) ปี และสัญญาอ้างอิงส่งมอบใน \(U=1.5\) ปี

ตัวอย่างนี้ระบุชัดว่า **จ่าย premium วันนี้ และชำระ payoff \((F(T,U)-K)^+\) เป็นเงินสดที่ T** จึงคิด variance ถึง T และ discount ถึงวันที่จ่าย T เมื่อให้อัตราดอกเบี้ย deterministic ราคา forward กับ futures ที่ส่งมอบวันเดียวกันจึงสอดคล้องกันภายใต้สมมติฐานมาตรฐาน แต่ราคานี้ไม่ใช่มูลค่าของการซื้อสินทรัพย์ที่จ่าย \(F_0\) วันนี้

สำหรับ Option ที่ใช้สิทธิแล้วได้ forward ซึ่งชำระที่ U หรือสัญญาแบบ futures-style margining ต้องกำหนดกระแสเงินสดและการชำระราคาใหม่ ก่อนเลือก discount factor ไม่ควรนำสูตรตัวอย่างนี้ไปใช้โดยเปลี่ยนชื่อสัญญาอย่างเดียว

In [6]:
def black76(F, strike, discount_factor, variance_to_expiry):
    if not all(math.isfinite(x) for x in (F, strike, discount_factor, variance_to_expiry)):
        raise ValueError("Inputs must be finite")
    if F <= 0 or strike <= 0 or discount_factor <= 0 or variance_to_expiry < 0:
        raise ValueError("Require positive prices/discount and nonnegative variance")
    if variance_to_expiry == 0:
        return dict(call=discount_factor*max(F-strike, 0.0),
                    put=discount_factor*max(strike-F, 0.0))
    root_v = math.sqrt(variance_to_expiry)
    b1 = (math.log(F/strike)+0.5*variance_to_expiry)/root_v
    b2 = b1-root_v
    return dict(call=discount_factor*(F*normal_cdf(b1)-strike*normal_cdf(b2)),
                put=discount_factor*(strike*normal_cdf(-b2)-F*normal_cdf(-b1)),
                d1=b1, d2=b2)

F0, futures_K, futures_r, futures_vol = 100.0, 100.0, .05, .20
option_expiry, contract_delivery = 1.0, 1.5
assert option_expiry <= contract_delivery
payment_discount = math.exp(-futures_r*option_expiry)
expiry_variance = futures_vol*futures_vol*option_expiry
black = black76(F0, futures_K, payment_discount, expiry_variance)
close(black["call"]-black["put"], payment_discount*(F0-futures_K))
close(black76(F0, futures_K, payment_discount, 0.0)["call"], 0.0)
close(black76(105.0, futures_K, payment_discount, 0.0)["call"], payment_discount*5.0)
# A lognormal forward under its expiry-forward measure has zero drift.
# With deterministic rates it is also driftless under the money-market Q measure.
quad_f_call, quad_f_put = payoff_quadrature(F0, futures_K, 0.0, 0.0, expiry_variance)
close(black["call"], payment_discount*quad_f_call, 1e-8)
close(black["put"], payment_discount*quad_f_put, 1e-8)
close(F0*math.exp(-0.5*expiry_variance)*math.exp(0.5*expiry_variance), F0)
incorrect_clock = black76(F0, futures_K, math.exp(-futures_r*contract_delivery),
                         futures_vol*futures_vol*contract_delivery)
print(f"Option expiry/payment T={option_expiry:.2f} years; underlying contract delivery U={contract_delivery:.2f} years")
print(f"Discount to T={payment_discount:.9f}; variance through T={expiry_variance:.9f}")
print(f"Black76 Call={black['call']:.9f}; Put={black['put']:.9f}")
print(f"Parity Call-Put={black['call']-black['put']:.9f}")
print(f"Incorrectly using U for both clocks gives Call={incorrect_clock['call']:.9f}")
print("Under the deterministic-rate model E_Q[F(T,U)]=F(0,U); F itself, not exp(-rT)*F, is the martingale here.")
print("Underlying delivery U affects the quoted F(0,U) and its volatility; it is not automatically the option's life.")

Option expiry/payment T=1.00 years; underlying contract delivery U=1.50 years
Discount to T=0.951229425; variance through T=0.040000000
Black76 Call=7.577082146; Put=7.577082146
Parity Call-Put=0.000000000
Incorrectly using U for both clocks gives Call=9.043341972
Under the deterministic-rate model E_Q[F(T,U)]=F(0,U); F itself, not exp(-rT)*F, is the martingale here.
Underlying delivery U affects the quoted F(0,U) and its volatility; it is not automatically the option's life.


## 7. สิ่งที่ตรวจผ่าน และขอบเขตของผล

การรันตรวจ benchmark, payoff decomposition, put–call parity, Radon–Nikodym identity แบบรายเส้นทาง, PDE ด้วย finite differences, สูตร deterministic term structure และ Black 76 เทียบกับ quadrature ส่วนผล Monte Carlo รายงาน estimate กับ standard error แยกกัน เพื่อไม่ให้ความคลาดเคลื่อนจากการสุ่มปะปนกับความคลาดเคลื่อนของโมเดล

ทุกผลอาศัยสมมติฐานที่ระบุในบทและข้อมูลจำลอง ไม่มีการสอบเทียบกับตลาดจริง ราคา Option คือ premium วันนี้ ไม่ใช่ payoff หรือกำไรสุทธิของผู้ซื้อ

In [7]:
# A small deterministic grid exercises calls, puts and negative-rate cases.
checked = 0
for stock in [70.0, 100.0, 140.0]:
    for integrated_rate in [-0.01, 0.04]:
        for integrated_dividend in [0.0, 0.03]:
            for integrated_variance in [0.0, 0.01, 0.09]:
                result = generalized_bs(stock, 100.0, integrated_rate,
                                        integrated_dividend, integrated_variance)
                stock_pv = stock*math.exp(-integrated_dividend)
                strike_pv = 100.0*math.exp(-integrated_rate)
                close(result["call"]-result["put"], stock_pv-strike_pv)
                assert -1e-10 <= result["call"] <= stock_pv+1e-10
                assert -1e-10 <= result["put"] <= strike_pv+1e-10
                assert result["call"] >= max(stock_pv-strike_pv, 0.0)-1e-10
                assert result["put"] >= max(strike_pv-stock_pv, 0.0)-1e-10
                checked += 1
print(f"Passed deterministic parity and price-bound checks for {checked} parameter combinations.")
print("All notebook code cells executed in order, with fixed simulation seeds.")

Passed deterministic parity and price-bound checks for 36 parameter combinations.
All notebook code cells executed in order, with fixed simulation seeds.
